# MuonClip angular WeightWatcher: initial / best / final

This notebook loads three **actual saved checkpoints** from the same one-head NanoGPT MuonClip run:

- `checkpoint_initial.pt` — step 0;
- `checkpoint_best.pt` — best validation checkpoint;
- `checkpoint_final.pt` — completed final checkpoint.

It performs the same angular power-law analysis for the three meaningful relative flows:

\[
W_{0}\to W_{\mathrm{best}},\qquad
W_{0}\to W_{\mathrm{final}},\qquad
W_{\mathrm{best}}\to W_{\mathrm{final}}.
\]

For every matrix and both angular sectors, each observed flow is compared with a matched Haar/Stiefel random-angular null.

## Power-law contract

Every positive projective angular value is passed to `powerlaw.Fit`:

```python
powerlaw.Fit(values, discrete=False, verbose=False)
```

No `xmin` and no `xmax` are supplied. The package chooses the start of the tail with its MLE/KS procedure, and **all largest values through the observed maximum remain in the fit**.

The notebook produces package-native PDF/CDF/CCDF plots, far-tail CCDF zooms against the random null, pairwise alpha plots with random-null confidence intervals, and a three-state radial spectrum plot.

## Command-line / Papermill environment

```bash
cd /path/to/rg_optimizers
export RG_OPTIMIZERS_ROOT="$PWD"
export RUNROOT=/tmp/<same-run-root-used-for-training>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RUN_DIR="$RESULTS_ROOT/$TARGET_OPTIMIZER/seed_$TARGET_SEED"

# Optional explicit checkpoint overrides
# export INITIAL_CHECKPOINT_PATH="$RUN_DIR/checkpoint_initial.pt"
# export BEST_CHECKPOINT_PATH="$RUN_DIR/checkpoint_best.pt"
# export FINAL_CHECKPOINT_PATH="$RUN_DIR/checkpoint_final.pt"

export ANGULAR_N_NULL=500
export ANGULAR_SHOW_PLOTS=1

papermill \
  baseline/nanogpt_one_head/notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb \
  /tmp/angular_initial_best_final_seed${TARGET_SEED}.out.ipynb
```


In [ ]:
from pathlib import Path
import os
import sys

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Set RG_OPTIMIZERS_ROOT or launch from the rg_optimizers repository"
    )

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)


In [ ]:
from IPython.display import display
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_three_checkpoint import run_three_checkpoint_analysis

CONFIG = AnalysisConfig.from_env()
print(CONFIG)
print("BEST_CHECKPOINT_PATH =", os.environ.get("BEST_CHECKPOINT_PATH", "<RUN_DIR>/checkpoint_best.pt"))


In [ ]:
RESULTS, MANIFEST = run_three_checkpoint_analysis(CONFIG)

print("Checkpoints:")
for state, path in MANIFEST["checkpoints"].items():
    print(f"  {state:7s} step={MANIFEST['steps'][state]:7d}  {path}")

display(RESULTS)

print("\nSummary CSV:")
run_dir = Path(MANIFEST["checkpoints"]["final"]).parent
output_dir = (
    Path(CONFIG.output_dir).expanduser().resolve()
    if CONFIG.output_dir
    else run_dir / "diagnostics" /
    f"angular_saved_step_0000000_vs_final_{MANIFEST['steps']['final']:07d}" /
    "initial_best_final"
)
print(output_dir / "angular_initial_best_final_powerlaw_summary.csv")
print("\nPlots are under:", output_dir)


## What to compare

For each matrix and `tilt`/`twist`, compare the three rows:

- `initial->best`
- `initial->final`
- `best->final`

The key numerical fields are `actual_alpha`, `actual_xmin`, `actual_D`, `actual_tail_n`, `actual_tail_decades`, and the random-null interval `null_alpha_2p5 ... null_alpha_97p5`.

The most important plots are:

```text
<MATRIX>_radial_initial_best_final.png
<MATRIX>_<tilt|twist>_pairwise_alpha_vs_random.png
<MATRIX>_<tilt|twist>_<state>_to_<state>_far_tail_zoom_ccdf.png
<MATRIX>__<state>_to_<state>_<tilt|twist>_powerlaw_pdf_loglog.png
<MATRIX>__<state>_to_<state>_<tilt|twist>_powerlaw_cdf.png
<MATRIX>__<state>_to_<state>_<tilt|twist>_powerlaw_ccdf_loglog.png
```

A power-law-looking tail is not evidence by itself. The trained tail must be distinguishable from the matched random-angular null, and its far-tail extent must survive through the largest observed angular values.
